In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [4]:
# 1. Dataset load karo
df = pd.read_csv("diabetic_data.csv")

In [5]:
df.describe()

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,6.078800e+04,6.078800e+04,60788.000000,60788.000000,60788.000000,60788.000000,60788.000000,60788.000000,60788.000000,60788.000000,60788.000000,60788.000000,60788.000000
mean,9.671185e+07,4.146263e+07,2.175380,4.131391,6.030911,4.528558,42.749655,1.348983,15.509360,0.256778,0.139189,0.595956,7.038462
std,4.709115e+07,3.559434e+07,1.572949,5.850999,4.588237,3.062870,19.083978,1.676017,8.170832,0.947908,0.646483,1.213157,2.010280
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,5.851407e+07,9.393350e+06,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,5.000000
50%,9.831163e+07,2.492085e+07,2.000000,1.000000,7.000000,4.000000,44.000000,1.000000,14.000000,0.000000,0.000000,0.000000,8.000000
75%,1.408804e+08,7.449455e+07,3.000000,5.000000,7.000000,6.000000,56.000000,2.000000,19.000000,0.000000,0.000000,1.000000,9.000000
max,1.701039e+08,1.152185e+08,8.000000,28.000000,20.000000,14.000000,129.000000,6.000000,81.000000,36.000000,42.000000,21.000000,9.000000


In [6]:
df.isnull().sum()

,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


In [7]:
df.shape


(60788, 50)

In [8]:
df = df.replace("?", "Unknown")

In [9]:
df["target"] = (df["readmitted"] == "<30").astype(int)

In [10]:
X = df[
    [
        "time_in_hospital",
        "num_lab_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses"
    ]
]

In [11]:
y = df["target"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
X_test

array([[-1.15411044, -1.82266071, -1.40882719, ..., -0.22027976,
        -0.49167751,  0.47850671],
       [-1.15411044,  0.37730241, -1.28647122, ..., -0.22027976,
        -0.49167751, -1.01522056],
       [-0.82754573,  0.63920278, -1.40882719, ..., -0.22027976,
         0.33009446,  0.47850671],
       ...,
       [-0.50098102,  1.63442419, -0.18526752, ..., -0.22027976,
        -0.49167751,  0.47850671],
       [-0.82754573,  0.796343  ,  0.42651232, ..., -0.22027976,
        -0.49167751, -1.01522056],
       [ 0.1521484 , -0.14649833,  3.73012345, ..., -0.22027976,
         0.33009446, -1.01522056]])

In [15]:
X_train

array([[-0.50098102, -0.19887841, -0.79704735, ..., -0.22027976,
        -0.49167751, -2.01103875],
       [ 0.47871311, -2.18932123, -0.18526752, ..., -0.22027976,
        -0.49167751, -1.01522056],
       [ 1.13184253,  1.47728397,  0.05944442, ..., -0.22027976,
        -0.49167751, -0.51731147],
       ...,
       [ 2.43810137,  1.79156442,  0.67122426, ..., -0.22027976,
        -0.49167751,  0.9764158 ],
       [-1.15411044, -2.13694115,  0.18180039, ..., -0.22027976,
        -0.49167751, -0.01940238],
       [-0.17441631,  1.0058633 , -0.06291155, ..., -0.22027976,
        -0.49167751,  0.47850671]])

In [16]:
model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000
)

In [17]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [18]:
y_prob = model.predict_proba(X_test)[:, 1]

In [22]:
auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

ROC-AUC: 0.5021989124446693


In [23]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = (y_prob >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[10724    42]
 [ 1389     3]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     10766
           1       0.07      0.00      0.00      1392

    accuracy                           0.88     12158
   macro avg       0.48      0.50      0.47     12158
weighted avg       0.79      0.88      0.83     12158



In [25]:
y_pred = (y_prob >= 0.5).astype(int)

print("Predictions:")
print(pd.Series(y_pred).value_counts())

Predictions:
0    12113
1       45
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')